In [1]:
import streamlit as st
import pandas as pd
from typing import List, Dict, Optional, Any
import plotly.express as px
import plotly.graph_objects as go

In [2]:
import pandas as pd
from pathlib import Path
from typing import List, Dict, Optional, Any

def read_data(file_path: str) -> (Dict[str, pd.DataFrame], List[str]):
    def load_sheets(file_path: str, sheet_names: List[str], header: int = 3) -> Dict[str, pd.DataFrame]:
        return {name: pd.read_excel(file_path, sheet_name=name, header=header) for name in sheet_names}
    
    sheet_names = pd.ExcelFile(file_path).sheet_names
    sheet_data = load_sheets(file_path, sheet_names[3:7])
    return sheet_data, sheet_names

def set_up_data():
    path_digiclass = "/Users/leonardhaas/code/streamlit/data/processed_data/added_digiclass.csv"
    #Path(__file__).parent / "data/processed_data/added_digiclass.csv"
    digiclass_data = pd.read_csv(path_digiclass,index_col=0)
    
    path_livingstone = "/Users/leonardhaas/code/streamlit/data/processed_data/isco_livingstone.csv"
    #Path(__file__).parent / "data/processed_data/isco_livingstone.csv"
    livingstone_data = pd.read_csv(path_livingstone, index_col=0)  # This was loading digiclass_data again
    
    merged_data = pd.merge(digiclass_data, livingstone_data, left_on='ISCO.Code', right_on='ISCO-Code')
    merged_data.drop(columns=['ISCO.Code'], inplace=True)
    return merged_data

In [13]:
livingstone_data = set_up_data()

In [14]:
# Only update the rows where Stellung.im.Beruf has specific values
# Change 'Selbstständige ohne Beschäftigte' to 'Selbstständige'
livingstone_data.loc[livingstone_data['Stellung.im.Beruf'] == 'Selbstständige ohne Beschäftigte', 'modifiziert_livingstone'] = 'Selbstständige'

# Keep 'Selbstständige mit Beschäftigten' as is 
# (this line isn't necessary if you don't need to change it, but included for clarity)
livingstone_data.loc[livingstone_data['Stellung.im.Beruf'] == 'Selbstständige mit Beschäftigten', 'modifiziert_livingstone'] = 'Selbstständige mit Beschäftigten'

# All other values in modifiziert_livingstone remain unchanged

livingstone_data_clean = livingstone_data.dropna(subset=["modifiziert_livingstone"])

In [31]:


# Define the specific order you want
fraktion_order = [
    "Selbstständige", 
    "Selbstständige mit Beschäftigten",  # Besitzer
    "Top Management", 
    "Mittleres Management", 
    "Anleitende Beschäftigte",  # Manager
    "Hochspezialisierte Beschäftigte", 
    "Industriearbeiter*innen", 
    "Dienstleistungsarbeiter*innen"  # Arbeiter*innenklasse
]

colors = [
    '#1D4E1F',  # Dark green - Military (distinct from the others)
    '#0A1F44',  # Dark navy blue - Managers (more distinct from the others)
    '#F5B461',  # Golden yellow - Professionals (brighter)
    '#00CED1',  # DarkTurquoise - Technicians (brighter for distinction)
    '#9B59B6',  # Medium purple - Clerical (distinct from light purple)
    '#FF6B6B',  # Coral red - Service workers
    '#4A90E2',  # Sky blue - Agricultural
    '#af005f',  # deep pink - Craft workers
    '#4B2F2F',  # dark brown - Plant operators (more distinct from yellow)
    '#F2C9B3'   # Soft peach - Elementary (lighter for distinction)
]


In [19]:
# Ensure all categories in your data are included in your order
# Before creating the chart, check what categories actually exist
existing_categories = livingstone_data["modifiziert_livingstone"].unique()
print("Categories in data:", existing_categories)

# Filter fraktion_order to only include categories that exist in your data
valid_fraktion_order = [cat for cat in fraktion_order if cat in existing_categories]


Categories in data: ['Top Management' 'Mittleres Management' 'Hochspezialisierte Beschäftigte'
 'Anleitende Beschäftigte' 'Dienstleistungsarbeiter*innen'
 'Industriearbeiter*innen' nan 'Selbstständige mit Beschäftigten'
 'Selbstständige']


In [48]:
# Create the bar chart with the custom order
fig = px.bar(
    livingstone_data,
    x="Anzahl",
    y="modifiziert_livingstone",
    color='major_group',
    orientation='h',
    hover_data={
        'Berufsgattung(ISCO-Stufe 4)': True,
        'Anzahl': True,
        'modifiziert_livingstone': False,
        'major_group': False
    },
    title='Klassenanalyse',
    color_discrete_sequence=colors,
    category_orders={"modifiziert_livingstone": fraktion_order}
)
# Style the silces of the bars with black borders and slight transparency
fig.update_traces(
    marker=dict(line=dict(width=0.5, color='black')),
    opacity=0.8
)

# Update layout
fig.update_layout(
    width=1200,  # Reduce width
    height=600, 
    yaxis={
            'categoryorder': 'array',
            'categoryarray': fraktion_order[::],
            'title': 'Klassen',  # Add a title for the y-axis
            'title_standoff': 25,  # Distance between the axis and its title
            'tickfont': {'size': 12},  # Font size for the tick labels
            'titlefont': {'size': 14, 'color': 'black'}  # Font for the axis title
    },
    margin=dict(l=250, r=150),  # Increased right margin for meta-category labels
    legend=dict(
        yanchor="top",     # Anchor point is at the top of the legend box
        y=0.50,            # Position very close to the top (0.99 of the way up)
        xanchor="right",   # Anchor point is at the right of the legend box
        x=1.59,            # Position very close to the right edge (0.99 of the way right)
        bgcolor="rgba(255,255,255,0.8)",  # Semi-transparent white background
        bordercolor="rgba(0,0,0,0.2)",    # Light gray border
        borderwidth=1                     # Thin border
    ),
    plot_bgcolor='rgba(255,255,255,1)',  # White background
    
    legend_title_text='ISCO-08 Hauptgruppen'
)

fig.show()
